In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect('database.sqlite')

In [ ]:
# Constants
YEARS = [2022, 2023, 2024, 2025]
CATEGORICAL_COLUMNS = [
    'attackComplexity',
    'attackVector',
    'availabilityImpact',
    'baseSeverity',
    'confidentialityImpact',
    'integrityImpact',
    'privilegesRequired',
    'scope',
    'userInteraction'
]

In [ ]:
# Basic data validation checks
dfs = []
for year in YEARS:
    try:
        df_year = pd.read_sql_query(f"SELECT * FROM YEAR_{year}", con)
        df_year['source_year'] = year  # Keep track of the year group
        dfs.append(df_year)
    except Exception as e:
        print(f"Warning: Could not read table YEAR_{year}: {e}")

db = pd.concat(dfs, ignore_index=True)
db.info()

In [ ]:
def display_yearly_breakdown(df, column_name):
    # crosstab puts the feature on rows and source_year on columns
    breakdown = pd.crosstab(df[column_name], df['source_year']).fillna(0).astype(int)

    # Replace top left corner with the column name for clarity
    breakdown.index.name = None
    breakdown.columns.name = column_name

    # Ensure all years are represented in the columns even if a table was empty
    for year in YEARS:
        if year not in breakdown.columns:
            breakdown[year] = 0
            
    # Sort columns chronologically
    breakdown = breakdown[sorted(breakdown.columns)]
    display(breakdown)

In [ ]:
for col in CATEGORICAL_COLUMNS:
    if col in db.columns:
        display_yearly_breakdown(db, col)

In [ ]:
# Only numeric columns are baseScore
score_breakdown = pd.crosstab(db['baseScore'], db['source_year']).fillna(0).astype(int)

# Replace top left corner with the column name for clarity
score_breakdown.index.name = None
score_breakdown.columns.name = 'baseScore'

for year in YEARS:
    if year not in score_breakdown.columns:
        score_breakdown[year] = 0
display(score_breakdown[sorted(score_breakdown.columns)])